# M03 · Losses & Optimization — Toy Example, Step by Tiny Step

**Companion to lesson M03. Written for someone new to ML.**

A **loss** turns a prediction into a single number to minimize. This notebook builds the two you meet most — **log loss** (for probabilities) and **squared/absolute/Huber** (for numbers) — one tiny step at a time, with a **break case** showing how one outlier hijacks squared error.

## Step 0 · Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)
plt.rcParams["figure.figsize"] = (6, 4)

def log(label, value):
    print(f"[{label}] {value}")

log("setup", "tools ready — seed fixed to 0")

## Step 1 · Log loss punishes confident-and-wrong predictions

For a true label `y=1`, log loss is `-log(p)` where `p` is the predicted probability of the true class. Being confident and **right** costs almost nothing; being confident and **wrong** costs a lot.

In [ ]:
ps = np.array([0.9, 0.5, 0.1, 0.01])          # predicted P(y=1) for a truly-positive example
for p in ps:
    log(f"y=1, p={p}", f"log loss = -log(p) = {(-np.log(p)):.3f}")
assert abs(-np.log(0.9) - 0.105) < 1e-2       # confident-right ~ 0.105
assert abs(-np.log(0.01) - 4.605) < 1e-2      # confident-wrong ~ 4.605 (44x worse)

grid = np.linspace(0.01, 0.99, 100)
plt.plot(grid, -np.log(grid), label="y=1: -log(p)")
plt.plot(grid, -np.log(1 - grid), label="y=0: -log(1-p)")
plt.title("log loss vs predicted probability"); plt.xlabel("p"); plt.ylabel("loss"); plt.legend(); plt.show()

▶ What you'll see: loss near 0 when confident-and-right, exploding when confident-and-wrong.

## Step 2 · Regression losses and the outlier break case

**MSE** squares the residual, **MAE** takes its absolute value, **Huber** is quadratic near 0 and linear far out. The break case: a single residual of 10 costs **100** under MSE but only **10** under MAE — so one outlier can dominate an MSE fit.

In [ ]:
r = np.linspace(-12, 12, 200); delta = 1.0
mse = r**2; mae = np.abs(r)
huber = np.where(np.abs(r) <= delta, 0.5*r**2, delta*(np.abs(r) - 0.5*delta))
log("residual 10 -> MSE", 10**2); log("residual 10 -> MAE", 10)
assert 10**2 > 10                              # MSE lets the outlier dominate

plt.plot(r, mse, label="MSE (r^2)"); plt.plot(r, mae, label="MAE (|r|)")
plt.plot(r, huber, label="Huber (delta=1)"); plt.ylim(0, 40)
plt.title("regression losses vs residual"); plt.xlabel("residual"); plt.ylabel("loss"); plt.legend(); plt.show()

▶ What you'll see: MSE's steep parabola (outlier-sensitive) vs MAE's V and Huber's blend.

## Recap

- **Log loss** = `-log(p_true)`: confident-wrong is punished hardest.
- **MSE** squares residuals (outlier-sensitive); **MAE** is robust; **Huber** blends both.
- Choosing the loss *is* choosing what mistakes you care about.